
### Pydantic V2 Assessment: AI Travel Agent Itinerary Validator


OBJECTIVE:
Build a Pydantic data validation schema to parse and validate travel itinerary
data produced by an AI Agent.

REQUIREMENTS:

1. Define a nested model `Flight`:
   - flight_number: str
   - price_usd: float (must be strictly greater than 0)
   - origin: str (must be exactly 3 characters long, e.g., "SFO")
   - destination: str (must be exactly 3 characters long, e.g., "JFK")
   - Include `Field(description=...)` metadata on all fields.
   - Add a `@field_validator` for `origin` and `destination` that automatically
     converts lowercase input (e.g., 'sfo') to uppercase ('SFO').

2. Define a parent model `Itinerary`:
   - trip_name: str
   - flights: List[Flight] (must contain at least 1 flight)
   - total_budget_usd: float
   - Include `Field(description=...)` metadata on all fields.
   - Add a custom validator (`@model_validator(mode='after')`) to verify that 
     the sum of all flight prices does NOT exceed `total_budget_usd`. 
     If it exceeds the budget, raise a ValueError.

3. Complete the execution code below to test both valid and invalid payloads.



## My solution

In [0]:
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator
from typing import List

In [0]:
class Flight(BaseModel):
    flight_number: str = Field(description="The official flight number")
    price_usd:float = Field(gt = 0, description = "the price of the trip in US dollars")
    origin:str = Field(min_length = 3, max_length =3, description = 'the actual International code of the origin Airport')
    destination:str = Field(gmin_length = 3, max_length =3, description = 'the actual International code of the destination Airport')

    @field_validator('origin', 'destination')
    @classmethod
    def convert_to_uppercase(cls, value:str) -> str:
        return value.upper()

In [0]:
f1 = {"flight_number": "BA101", "price_usd": 450.00, "origin": "sfo", "destination": "lhr"}

In [0]:
flight_s1 = Flight.model_validate(f1)
print(flight_s1)

In [0]:
class Itinerary(BaseModel):
    trip_name:str = Field(descripton = "the name of the trip")
    flights:List[Flight] = Field(min_length = 1 ,descripton = "the list of the flights taken or used")
    total_budget_usd:float = Field(descripton = "the total budget for the trip")

    @model_validator(mode='after')
    def validate_total_budget(self):
        prices_sum = sum([f.price_usd for f in self.flights])
        if prices_sum > self.total_budget_usd:
            raise ValueError(f"Total flight cost ${prices_sum} exceeds budget ${self.total_budget_usd}")
        return self



In [0]:
valid_json = {
    "trip_name": "European Tour",
    "total_budget_usd": 1500.00,
    "flights": [
        {"flight_number": "BA101", "price_usd": 450.00, "origin": "sfo", "destination": "lhr"},
        {"flight_number": "AF202", "price_usd": 350.00, "origin": "LHR", "destination": "cdg"}
    ]
}

In [0]:
it1 = Itinerary.model_validate(valid_json)
print(it1)

In [0]:
over_budget_json = {
    "trip_name": "Quick Business Trip",
    "total_budget_usd": 500.00,
    "flights": [
        {"flight_number": "DL88", "price_usd": 900.00, "origin": "JFK", "destination": "SFO"}
    ]
}

In [0]:
it1 = Itinerary.model_validate(over_budget_json)
print(it1)